**1. Create a bronze table that ingests raw sales data as-is (no transformations), preserving all original
columns plus an ingestion timestamp.**

In [0]:
%sql
CREATE OR REPLACE TABLE dev.bronze.sales_data_raw AS
SELECT *
FROM read_files(
    "/Volumes/dev/bronze/raw/sales.csv",
    header => true,
    inferSchema => true
)

**2. Build a silver table from bronze that removes duplicates, fixes data types, and drops clearly invalid
rows.**

In [0]:
df = spark.read.table("dev.bronze.sales_data_raw")

In [0]:
from pyspark.sql import functions as F
df_cleaned = df.drop("_rescued_data") \
    .dropDuplicates().dropna(how="all") \
    .withColumn("order_id", F.col("order_id").cast("string")) \
    .withColumn("customer_id", F.col("customer_id").cast("string")) \
    .withColumn("transaction_id", F.col("transaction_id").cast("string")) \
    .withColumn("product_id", F.col("product_id").cast("string"))

df_cleaned.write.mode("overwrite").saveAsTable("dev.silver.sales_data_cleaned")


**3. Build a gold table that aggregates silver into a business-ready view (e.g., daily revenue by store).**

In [0]:
%sql
-- Average Order Value Per Month
SELECT 
    DATE_TRUNC('month', order_date) AS order_month,
    ROUND(SUM(total_amount) / COUNT(DISTINCT order_id), 2) AS average_order_value
FROM 
    dev.silver.sales_data_cleaned
GROUP BY 
    DATE_TRUNC('month', order_date)
ORDER BY 
    order_month DESC;

## 2. Intermediate Tasks

**4. Diagram your bronze/silver/gold pipeline and label, for each layer, who the primary consumer is
(engineers, analysts, executives).**

![](/Volumes/dev/bronze/raw/job.png)

I have given the pipeline image above where each layer is shown as bronze, silver and gold.
- The bronze layer we just ingest the raw data, and save as a table in the bronze table `dev.bronze.sales_job` and this layer was besically for engineers.

- The silver layer, is where the data is cleaned, I have cleaned the raw data from the bronze table then stored it in the silver table `dev.silver.sales_cleaned_job` and for this layer the primary consumer is analysts.

- The gold layer, is where the business KPI's is created. According to the business needs, I have created different KPI's using SQL queries
`dev.gold.top_5_customers_job`, and this layer is specifically for executives.

**5. Recreate one part of your silver transformation using Lakeflow Designer's visual, no-code interface
and compare the experience to writing it in code.**

/Workspace/Users/surajitm0nd0l9732@gmail.com/Drafts/Visual data prep 2026-08-25 18:52:18.designer.ipynb

I have created a Lakeflow Designer's Visual and performed the same above task. In the lakeflow designer it easier to do it you just need to use proper oparetor for the exact task, for each task you need a operator and you don't need you write much code only some time if you need to any SQL query in the formual part. So it is easier to implement compaired to the writing the Entire Bronze load, then cleaning in silver.

**6. Chain bronze → silver → gold as a Lakeflow Job with proper task dependencies, and configure it to
run on a schedule.**

I have created the three different notebook 
- bronze: /Workspace/Users/surajitm0nd0l9732@gmail.com/Data_Engineering_Assignment_Cyntexa/day_6_assignment/bronze
- silver: /Workspace/Users/surajitm0nd0l9732@gmail.com/Data_Engineering_Assignment_Cyntexa/day_6_assignment/silver
- gold: /Workspace/Users/surajitm0nd0l9732@gmail.com/Data_Engineering_Assignment_Cyntexa/day_6_assignment/gold

and then created the job where in the job first task the bronze notebook, then the silver notebook and then the gold notebook, then I have added the triger and scheduled the job. I have given the yml file for the job I have created below.

```
resources:
  jobs:
    New_Job_2026_08_25_23_02_42:
      name: New Job 2026-08-25 23:02:42
      schedule:
        quartz_cron_expression: 26 59 23 * * ?
        timezone_id: Asia/Calcutta
        pause_status: UNPAUSED
      tasks:
        - task_key: bronze
          notebook_task:
            notebook_path: /Workspace/Users/surajitm0nd0l9732@gmail.com/Data_Engineering_Assignment_Cyntexa/day_6_assignment/bronze
            source: WORKSPACE
          min_retry_interval_millis: 900000
          disable_auto_optimization: true
        - task_key: silver
          depends_on:
            - task_key: bronze
          notebook_task:
            notebook_path: /Workspace/Users/surajitm0nd0l9732@gmail.com/Data_Engineering_Assignment_Cyntexa/day_6_assignment/silver
            source: WORKSPACE
          min_retry_interval_millis: 900000
          disable_auto_optimization: true
        - task_key: gold
          depends_on:
            - task_key: silver
          notebook_task:
            notebook_path: /Workspace/Users/surajitm0nd0l9732@gmail.com/Data_Engineering_Assignment_Cyntexa/day_6_assignment/gold
            source: WORKSPACE
          min_retry_interval_millis: 900000
          disable_auto_optimization: true
      queue:
        enabled: true
      parameters:
        - name: catalog
          default: dev
      performance_target: PERFORMANCE_OPTIMIZED
```


## 3. Advanced Tasks

**7. Extend the gold layer with a second aggregation for a different stakeholder (e.g., the inventory team)
and justify why it belongs in gold rather than being computed ad hoc by that team.**

In [0]:
%sql
CREATE OR REPLACE VIEW dev.gold.product_inventory_summary AS
SELECT
    product_id,
    SUM(quantity) AS total_units_sold,
    COUNT(DISTINCT order_id) AS number_of_orders,
    MAX(order_date) AS last_sale_date
FROM dev.silver.sales_cleaned_job
GROUP BY
    product_id
ORDER BY total_units_sold DESC;


The inventory team could technically calculate these metrics themselves from the Silver layer, but keeping the aggregation in Gold is better because:
- Reusable: The same inventory metrics can be used by dashboards, reports, and analysts without everyone recreating the logic.
- Consistent: Everyone uses the same definition of metrics such as total_units_sold.
- Performance: The aggregation is computed once rather than repeatedly by different users.
- Business-ready: Gold contains data that has already been cleaned, transformed, and aggregated for a specific business purpose.
- Governance: Centralizing the business logic makes it easier to maintain and audit.
- Faster reporting: The inventory team can query the prepared Gold table instead of processing large amounts of detailed sales data every time.

**8. Write a short design note on which parts of this pipeline should run in the customer's data plane vs.
rely on Databricks' control plane, and what that means for a network/security review.**

**Customer Data Plane**

The following should run in the customer's data plane:

* **Data ingestion:** Read raw `sales.csv` files.
* **Bronze layer:** Store raw/ingested data.
* **Silver layer:** Clean and transform the data.
* **Gold layer:** Create business aggregations such as Top 5 Customers and Inventory Summary.
* **Data processing:** Spark compute that directly processes customer data.
* **Data storage:** Customer data stored in the customer's cloud storage.

**Databricks Control Plane**

The control plane should mainly handle:

* Workspace management
* Job scheduling and orchestration
* User, group, and permission management
* Cluster/job configuration
* APIs and platform management
* Monitoring and control operations

**Network & Security Review**

The security team should review:

* **Network connectivity** between the control plane and data plane.
* Whether **private connectivity** is required and properly configured.
* Required **ports, endpoints, and firewall rules**.
* **Encryption in transit and at rest**.
* **Authentication, authorization, and RBAC**.
* Secure management of **secrets and credentials**.
* **Audit logging and monitoring**.
* **Data residency and compliance** requirements.
* What customer data, if any, crosses from the data plane to the control plane.

**In short:** The **data plane processes and stores customer data**, while the **control plane manages and orchestrates the Databricks environment**. The network/security review ensures that communication between them is secure, controlled, encrypted, and compliant.


**9. (Data Analyst) Build a query or lightweight dashboard directly against the gold table, and identify
one data-quality issue you can trace back to a specific bronze or silver transformation decision.**

In [0]:
region_df = spark.read.csv(
    "/Volumes/dev/bronze/raw/dim_store_regions.csv",
    header=True,
    inferSchema=True
)
e_commerce_df = spark.read.csv(
    "/Volumes/dev/bronze/raw/ecommerce_raw_transactions.csv",
    header=True,
    inferSchema=True
)
region_df.write.mode("overwrite").saveAsTable("dev.bronze.region_dim")
e_commerce_df.write.mode("overwrite").saveAsTable("dev.bronze.ecommerce_raw")

In [0]:
from pyspark.sql.functions import *

region_df2 = spark.read.table("dev.bronze.region_dim")
e_commerce_df2 = spark.read.table("dev.bronze.ecommerce_raw")

region_cleaned = region_df2.dropDuplicates().dropna(how="all") 
e_commerce_cleaned = e_commerce_df2.dropDuplicates().dropna(how="all") \
    .withColumn("store_id", trim(col("store_id")))

region_cleaned.write.mode("overwrite").saveAsTable("dev.silver.region_cleaned")
e_commerce_cleaned.write.mode("overwrite").saveAsTable("dev.silver.ecommerce_cleaned")


In [0]:
%sql
select * from dev.silver.ecommerce_cleaned

In [0]:
%sql
select * from dev.silver.region_cleaned

In [0]:
%sql
SELECT store_id, LEN(store_id) AS store_id_length
FROM dev.silver.ecommerce_cleaned

In [0]:
%sql
SELECT store_id, LEN(store_id) AS store_id_length
FROM dev.silver.region_cleaned

In [0]:
%sql
-- Daily Net Revenue by Region
CREATE OR REPLACE TABLE dev.gold.daily_net_revenue_by_region AS
SELECT 
    t.order_date,
    COALESCE(r.region_name, 'Unknown') AS region_name,
    ROUND(SUM(t.net_revenue), 2) AS total_net_revenue,
    COUNT(DISTINCT t.transaction_id) AS total_orders
FROM 
    dev.silver.ecommerce_cleaned t
LEFT JOIN 
    dev.silver.region_cleaned r 
ON 
    t.store_id = r.store_id
GROUP BY 
    t.order_date,
    COALESCE(r.region_name, 'Unknown')
ORDER BY 
    t.order_date;

In [0]:
%sql
select * from dev.gold.daily_net_revenue_by_region

![](/Volumes/dev/bronze/raw/graphical_representation.png)
![](/Volumes/dev/bronze/raw/unkonwn_region.png)

So I have created the dashboard, but in the dashboard I can see a region called unknown, which should not be there, because in the region table there is only three region Asis Pacific, Europe and North America.

So that's why I checked the silver table and also the gold table `dev.gold.daily_net_revenue_by_region`, and found that for the date that it is showing Unknown, on that day there is a valid order there in the table `dev.silver.ecommerce_cleaned` and the `store_id` also matches with the `dev.silver.region_cleaned`. 

For exmaple on the date `2026-08-11` there is two order and one of them is showing Unknow region showing in the table `dev.gold.daily_net_revenue_by_region`, but on that day 
there is two order also in the table `dev.silver.ecommerce_cleaned` which is showing currect data, and also `store_id` on those days matches with the `store_id` in the `dev.silver.region_cleaned` table. 

So the error must be in the `store_id`, but I can't find it with naked eye's, so then I have checked the length of the `store_id` in the table and I find out that the length of the word is bigger than actual word, means there there blank space there in the `store_id` which I should have trimmed in the silver. So now I have trimmed the extra blank space and now in the dashboard it showing the currect results, and it doesn't show the Unknown region. 

So below I have given the updated graph, of revenue by region and also the unknow region details table is empty.
![](/Volumes/dev/bronze/raw/cleaned_graph.png)